In [18]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")


In [19]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [20]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [21]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [26]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.agents import create_agent  # ← back to this
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    "llama-3.1-8b-instant",  # ← official replacement for the decommissioned model
    model_provider="groq",
    temperature=0,  # ← MUST be 0 for reliable tool calling
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt,
    checkpointer=InMemorySaver(),
)


In [27]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [28]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='12cc01d5-a2bc-4d94-b256-ebb60778421b'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'e0zszxzwe', 'function': {'arguments': '{"query":"langchain-mcp-adapters library"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 355, 'total_tokens': 375, 'completion_time': 0.047824359, 'completion_tokens_details': None, 'prompt_time': 0.037221449, 'prompt_tokens_details': None, 'queue_time': 0.040283321, 'total_time': 0.085045808}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f39ed-10f1-7e02-8dab-96748806676d-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'e0z

## Online MCP

In [29]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [34]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.agents import create_agent  # ← back to this
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    "llama-3.1-8b-instant",  # ← official replacement for the decommissioned model
    model_provider="groq",
    temperature=0,  # ← MUST be 0 for reliable tool calling
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt,
)

In [35]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='07ecc5e4-7d6d-46d1-9a48-0386561a26be'),
              AIMessage(content="I'm sorry, I can only answer questions about LangChain, LangGraph and LangSmith.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 564, 'total_tokens': 584, 'completion_time': 0.023208612, 'completion_tokens_details': None, 'prompt_time': 0.042558607, 'prompt_tokens_details': None, 'queue_time': 0.036170196, 'total_time': 0.065767219}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f39f0-dc25-7252-8aee-6d21d2315df5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 564, 'output_tokens': 20, 'total_tokens': 584})]}
